In [1]:
import pyspark

In [2]:
print(pyspark.__version__)

4.3.0


In [3]:
from pyspark.sql import SparkSession

In [4]:
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("first_look")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/16 19:04:09 WARN Utils: Your hostname, DESKTOP-8DK3P4V, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/16 19:04:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 19:04:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/16 19:04:12 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
df_raw = spark.read.csv("data/raw/yellow_tripdata_2021-01.csv.gz")

In [6]:
df_raw.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _c16: string (nullable = true)
 |-- _c17: string (nullable = true)



In [7]:
df_raw.show(3)

+--------+--------------------+--------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+--------------------+------------+--------------------+
|     _c0|                 _c1|                 _c2|            _c3|          _c4|       _c5|               _c6|         _c7|         _c8|         _c9|       _c10| _c11|   _c12|      _c13|        _c14|                _c15|        _c16|                _c17|
+--------+--------------------+--------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+--------------------+------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_date...|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surch...|total_a

In [8]:
import time

start = time.time()
df_inferred = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("data/raw/yellow_tripdata_2021-01.csv.gz")
)
print(f"Seconds: {time.time() - start:.1f}")
df_inferred.printSchema()

Seconds: 14.7
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [9]:
from pyspark.sql import types as T

yellow_schema = T.StructType([
    T.StructField("VendorID", T.IntegerType()),
    T.StructField("tpep_pickup_datetime", T.TimestampType()),
    T.StructField("tpep_dropoff_datetime", T.TimestampType()),
    T.StructField("passenger_count", T.IntegerType()),
    T.StructField("trip_distance", T.DoubleType()),
    T.StructField("RatecodeID", T.IntegerType()),
    T.StructField("store_and_fwd_flag", T.StringType()),
    T.StructField("PULocationID", T.IntegerType()),
    T.StructField("DOLocationID", T.IntegerType()),
    T.StructField("payment_type", T.IntegerType()),
    T.StructField("fare_amount", T.DoubleType()),
    T.StructField("extra", T.DoubleType()),
    T.StructField("mta_tax", T.DoubleType()),
    T.StructField("tip_amount", T.DoubleType()),
    T.StructField("tolls_amount", T.DoubleType()),
    T.StructField("improvement_surcharge", T.DoubleType()),
    T.StructField("total_amount", T.DoubleType()),
    T.StructField("congestion_surcharge", T.DoubleType()),
])

start = time.time()
df = (
    spark.read
    .option("header", True)
    .schema(yellow_schema)
    .csv("data/raw/yellow_tripdata_2021-01.csv.gz")
)
print(f"Seconds: {time.time() - start:.2f}")
df.printSchema()

Seconds: 0.19
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [10]:
from pyspark.sql import functions as F

start = time.time()
df_long = (
    df.filter(F.col("trip_distance") > 2)
      .select("PULocationID", "trip_distance","total_amount")
      .withColumn("amount_per_mile", F.col("total_amount") / F.col("trip_distance"))
)
print(f"Transformations took: {time.time() - start:.3f} seconds")

Transformations took: 0.379 seconds


In [11]:
df_long.explain()

== Physical Plan ==
*(1) Project [PULocationID#151, trip_distance#148, total_amount#160, (total_amount#160 / trip_distance#148) AS amount_per_mile#163]
+- *(1) Filter (isnotnull(trip_distance#148) AND (trip_distance#148 > 2.0))
   +- FileScan csv [trip_distance#148,PULocationID#151,total_amount#160] Batched: false, DataFilters: [isnotnull(trip_distance#148), (trip_distance#148 > 2.0)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/velpu/projects/data-engineering-project1/06-batch/data/raw/..., PartitionFilters: [], PushedFilters: [IsNotNull(trip_distance), GreaterThan(trip_distance,2.0)], ReadSchema: struct<trip_distance:double,PULocationID:int,total_amount:double>




In [12]:
for i in range(2):
    start = time.time()
    print(df_long.count())
    print(f"Action {i + 1} took: {time.time() - start:.1f} seconds")

566077
Action 1 took: 9.3 seconds


566077
Action 2 took: 4.1 seconds


In [13]:
print("Partitions:", df.rdd.getNumPartitions())
df_4 = df.repartition(4)
print("After repartition:", df_4.rdd.getNumPartitions())

Partitions: 1


After repartition: 4


In [14]:
start = time.time()
df_4.write.mode("overwrite").parquet("data/pq/yellow/2021/01")
print(f"Write took: {time.time() - start:.1f} seconds")

Write took: 48.3 seconds


In [15]:
df_pq = spark.read.parquet("data/pq/yellow/2021/01")
df_pq.printSchema()
print("Partitions:", df_pq.rdd.getNumPartitions())
print("Rows:", df_pq.count())

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)

Partitions: 2
Rows: 1369765


In [16]:
df_4.coalesce(2).write.mode("overwrite").parquet("data/pq/yellow_coalesced/2021/01")

In [19]:
(df.filter(F.col("trip_distance") > 2).groupBy("PULocationID").agg(F.count("*").alias("trips"), F.round(F.sum("total_amount"), 2).alias("revenue"),).filter(F.col("trips")> 10000).orderBy(F.col("revenue").desc()).limit(5).show()) 


AssertionError: 

In [20]:
from pyspark.sql import SparkSession, functions as F, types as T
import time

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("dataframes")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

df = spark.read.parquet("data/pq/yellow/2021/01")
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)



In [21]:
df_small = (
        df.select("tpep_pickup_datetime", "PULocationID", "DOLocationID", "trip_distance", "total_amount")
          .filter(F.col("total_amount") > 0)
          .withColumnRenamed("PULocationID", "pickup_location_id")
          .withColumnRenamed("DOLocationID", "dropoff_location_id")
          
)
df_small.show(10)

+--------------------+------------------+-------------------+-------------+------------+
|tpep_pickup_datetime|pickup_location_id|dropoff_location_id|trip_distance|total_amount|
+--------------------+------------------+-------------------+-------------+------------+
| 2021-01-14 17:54:16|               143|                142|          0.5|       11.15|
| 2021-01-07 11:42:51|                41|                 42|         1.84|         9.8|
| 2021-01-05 09:38:56|               170|                232|         3.14|        14.8|
| 2021-01-24 10:45:32|               237|                238|         1.52|       12.96|
| 2021-01-15 17:58:00|                48|                 48|         0.98|        18.5|
| 2021-01-05 19:54:54|               140|                263|         1.09|       12.36|
| 2021-01-13 14:04:46|               140|                262|         0.71|       10.79|
| 2021-01-05 19:17:53|               262|                 74|         1.25|       11.76|
| 2021-01-06 17:19:00

In [ ]:
df_enriched = (df_small.withColumn("pickup_date", F.to_date("tpep_pickup_datetime"))
                       .withColumn("distance_bucket", 
                                   F.when(F.col("trip_distance").isNull(), "unknown")
                                    .when(F.col("trip_distance") < 2, "short")
                                    .when(F.col("trip_distance") > 10, "medium")
                                    .otherwise("long")))
df_enriched.show(10)

+--------------------+------------------+-------------------+-------------+------------+-----------+---------------+
|tpep_pickup_datetime|pickup_location_id|dropoff_location_id|trip_distance|total_amount|pickup_date|distance_bucket|
+--------------------+------------------+-------------------+-------------+------------+-----------+---------------+
| 2021-01-14 17:54:16|               143|                142|          0.5|       11.15| 2021-01-14|          short|
| 2021-01-07 11:42:51|                41|                 42|         1.84|         9.8| 2021-01-07|          short|
| 2021-01-05 09:38:56|               170|                232|         3.14|        14.8| 2021-01-05|           long|
| 2021-01-24 10:45:32|               237|                238|         1.52|       12.96| 2021-01-24|          short|
| 2021-01-15 17:58:00|                48|                 48|         0.98|        18.5| 2021-01-15|          short|
| 2021-01-05 19:54:54|               140|                263|   

In [ ]:
daily = (
    df_enriched
    .groupBy("pickup_date", "distance_bucket")
    .agg(
        F.count("*").alias("trips"),
        F.round(F.sum("total_amount"), 2).alias("revenue"),
    )
    .orderBy("pickup_date", "distance_bucket")
)
daily.show(12)

+-----------+---------------+-----+---------+
|pickup_date|distance_bucket|trips|  revenue|
+-----------+---------------+-----+---------+
| 2009-01-01|           long|    1|     14.3|
| 2009-01-01|          short|    2|     22.1|
| 2020-12-31|           long|   10|   199.16|
| 2020-12-31|         medium|    2|     96.4|
| 2020-12-31|          short|    4|    44.02|
| 2021-01-01|           long|10168|212836.56|
| 2021-01-01|         medium| 2111|116349.78|
| 2021-01-01|          short|12339|140528.34|
| 2021-01-02|           long|13156|281947.64|
| 2021-01-02|         medium| 2785|159920.68|
| 2021-01-02|          short|18171|211333.37|
| 2021-01-03|           long| 9536| 205044.7|
+-----------+---------------+-----+---------+
only showing top 12 rows


In [ ]:
def distance_bucket(d):
    if d is None:
        return "unknown"
    return "short" if d < 2 else "medium" if d < 10 else "long"

distance_bucket_udf = F.udf(distance_bucket, returnType=T.StringType())

start = time.time()
df_small.withColumn("bucket", distance_bucket_udf("trip_distance")).groupBy("bucket").count().collect()
print(f"Python UDF: {time.time() - start:.1f} s")

start = time.time()
df_enriched.groupBy("distance_bucket").count().collect()
print(f"Built-in F.when: {time.time() - start:.1f} s")

/home/velpu/projects/data-engineering-project1/06-batch/.venv/lib/python3.13/site-packages/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


Python UDF: 8.8 s


Built-in F.when: 1.8 s


In [22]:
spark.stop()

In [23]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[2]").config("spark.driver.memory", "2g").getOrCreate()

green = spark.read.parquet("data/pq/green/*/*")
yellow = spark.read.parquet("data/pq/yellow/*/*")
print("green rows:", green.count())
print("yellow rows:", yellow.count())
spark.stop()

26/11/16 19:07:39 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: data/pq/green/*/*.
java.io.FileNotFoundException: File data/pq/green/*/* does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:401)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:209)
	at org.apache.spark.sql.catalyst.analysis.ResolveData

green rows: 2304517
yellow rows: 39649199


In [1]:
spark.stop()

NameError: name 'spark' is not defined